# V2-05 — Per-topic **trajectory** cartography (descriptive trend projection)

> **Honest framing — read first.** This is **descriptive trend projection, clearly labeled exploratory — NOT a predictive claim**, and **NOT the dead F3 GNN emergence classifier** (signed **NULL at Gate G4**: sealed test AUC 0.781 < 0.804 baseline, H1 FAIL). We do something far humbler here: we **extrapolate where each topic's annual share / volume has been trending**, with **stated uncertainty bands**. No graph, no classifier, no emergence label — just a per-topic time-series fit carried forward.

**Headline metric = topic *share*** of annual assigned output (the topic's papers ÷ all assigned papers that year). Share **removes the corpus-growth confound**: the 78-journal corpus grows over time, so a topic can rise in *raw volume* purely because the whole corpus grew. Share asks the sharper question — *is this topic gaining or losing ground relative to everything else?* **Raw volume is secondary** and shown only to make the confound visible.

**What this notebook renders**
1. Setup + load the two trajectory tables; print shapes, `fit_ok`, model & direction counts.
2. **Fan charts** — observed share (solid) + projected mean 2026–2030 (dashed) + the `share_lo`/`share_hi` band (shaded): a small-multiples grid of the strongest rising / falling topics, plus standalone fans for the headline rising and falling topics.
3. **Share-vs-volume** panel for one illustrative topic — the corpus-growth confound the share metric removes.
4. **Rising / falling leaderboards** (top-15 each).
5. Summary + caveats.

The fits live in `scifield.cartography.trajectory` and were built by `V2/scripts/build_trajectory.py`; this notebook does **only I/O + plotting** and **holds no `record_run`** (scripts own provenance — house convention). Tables are read from `V2/data_v2/trajectory/`.

## 1. Setup + load the trajectory tables

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — this notebook lives at V2/notebooks/, code is under src/.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

TRAJ_DIR = repo_root / "V2" / "data_v2" / "trajectory"
FIG_DIR = repo_root / "V2" / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

GRAIN = "leaf"  # 149 leaf topics; only grain present in the trajectory tables.

summary = pd.read_parquet(TRAJ_DIR / "trajectory_summary.parquet")
series = pd.read_parquet(TRAJ_DIR / "trajectory_series.parquet")
summary = summary[summary["grain"] == GRAIN].reset_index(drop=True)
series = series[series["grain"] == GRAIN].reset_index(drop=True)

print(f"summary: {summary.shape}  |  series: {series.shape}")
print(
    f"  fit_ok: {int(summary['fit_ok'].sum())}/{len(summary)} " f"({summary['fit_ok'].mean():.0%})"
)
print(f"  model:     {summary['model'].value_counts().to_dict()}")
print(f"  direction: {summary['direction'].value_counts().to_dict()}")

_obs_yrs = sorted(series.loc[series["kind"] == "observed", "year"].unique())
_proj_yrs = sorted(series.loc[series["kind"] == "projected", "year"].unique())
print(f"  observed years:  {_obs_yrs[0]}-{_obs_yrs[-1]} (last complete = {_obs_yrs[-1]})")
print(f"  projected years: {_proj_yrs[0]}-{_proj_yrs[-1]}")

# Colour key reused across figures.
RISE_C = "#34a853"  # green = rising
FALL_C = "#ea4335"  # red   = falling
FLAT_C = "#9aa0a6"  # grey  = flat
DIR_C = {"rising": RISE_C, "falling": FALL_C, "flat": FLAT_C}
OBS_C = "#1a73e8"  # observed line
PROJ_C = "#7b1fa2"  # projected line + band

summary: (149, 18)  |  series: (5352, 10)
  fit_ok: 149/149 (100%)
  model:     {'state_space_llt': 146, 'loglinear_fallback': 3}
  direction: {'flat': 81, 'falling': 38, 'rising': 30}
  observed years:  1995-2025 (last complete = 2025)
  projected years: 2026-2030


## 2. Fan charts — observed share + projected band

The `fan_chart(ax, topic_id)` helper draws one topic: **observed** `share` as a solid line+markers over the observed years (1995–2025); the **projected** mean (2026–2030) as a **dashed** line; and the **CI band** shaded between `share_lo` and `share_hi`. Bands are `NaN` on observed rows and populated only on the projected rows. To keep the projection visually continuous we **anchor the dashed line at the last observed point (2025)** before the first projected year — note the state-space model's projection level can start slightly off the last observation, which is expected (the smoother's filtered level ≠ the last noisy point).

Title = topic `label` (top-6 words) + `direction`.

In [2]:
SUM_IDX = summary.set_index("topic_id")


def fan_chart(ax, topic_id, *, show_legend=False, compact=False):
    """Plot one topic's observed share (solid) + projected share band (dashed+shaded)."""
    row = SUM_IDX.loc[topic_id]
    sub = series[series["topic_id"] == topic_id].sort_values("year")
    obs = sub[sub["kind"] == "observed"]
    proj = sub[sub["kind"] == "projected"]

    # Observed: solid line + markers.
    ax.plot(
        obs["year"],
        obs["share"],
        "-o",
        color=OBS_C,
        lw=1.6,
        ms=3.0,
        label="observed",
        zorder=3,
    )

    # Anchor the projection at the last observed point so the dashed line is continuous.
    if len(obs):
        last = obs.iloc[-1]
        proj_yr = np.concatenate([[last["year"]], proj["year"].to_numpy()])
        proj_mean = np.concatenate([[last["share"]], proj["share"].to_numpy()])
        band_yr = np.concatenate([[last["year"]], proj["year"].to_numpy()])
        band_lo = np.concatenate([[last["share"]], proj["share_lo"].to_numpy()])
        band_hi = np.concatenate([[last["share"]], proj["share_hi"].to_numpy()])
    else:  # pragma: no cover - every topic has observed rows here
        proj_yr = proj["year"].to_numpy()
        proj_mean = proj["share"].to_numpy()
        band_yr, band_lo, band_hi = (
            proj_yr,
            proj["share_lo"].to_numpy(),
            proj["share_hi"].to_numpy(),
        )

    # Projected: dashed mean + shaded CI band.
    ax.plot(proj_yr, proj_mean, "--", color=PROJ_C, lw=1.6, label="projected", zorder=3)
    ax.fill_between(
        band_yr,
        band_lo,
        band_hi,
        color=PROJ_C,
        alpha=0.18,
        lw=0,
        label="projection band",
        zorder=1,
    )
    # Visual divider at the last complete observed year.
    ax.axvline(obs["year"].max(), color="#bbbbbb", ls=":", lw=0.8, zorder=0)
    ax.set_ylim(bottom=0)

    direction = row["direction"]
    swatch = DIR_C.get(direction, FLAT_C)
    words = ", ".join(str(row["label"]).split(", ")[:4])
    if compact:
        ax.set_title(
            f"#{topic_id} {words}\n{direction} | slope {row['slope_share_per_yr']:+.5f}/yr",
            fontsize=8,
            color=swatch,
        )
    else:
        ax.set_title(
            f"#{topic_id}  {row['label']}\n"
            f"{direction}  |  slope {row['slope_share_per_yr']:+.5f}/yr  |  {row['model']}",
            fontsize=9,
            color=swatch,
        )
    ax.set_xlabel("year", fontsize=8)
    ax.set_ylabel("share of annual assigned output", fontsize=8)
    ax.tick_params(labelsize=7)
    if show_legend:
        ax.legend(loc="best", fontsize=7, frameon=False)
    return ax


# Data-driven selection — stays correct if numbers shift.
rising = summary.sort_values("slope_share_per_yr", ascending=False).head(4)
falling = summary.sort_values("slope_share_per_yr", ascending=True).head(4)
print("strongest RISING (by slope_share_per_yr):")
print(rising[["topic_id", "label", "slope_share_per_yr", "model"]].to_string(index=False))
print()
print("strongest FALLING (by slope_share_per_yr):")
print(falling[["topic_id", "label", "slope_share_per_yr", "model"]].to_string(index=False))

strongest RISING (by slope_share_per_yr):
 topic_id                                                           label  slope_share_per_yr              model
        2              health, information, data, care, digital, learning            0.003312 loglinear_fallback
       13 covid19, influenza, sarscov2, coronavirus, vaccine, respiratory            0.001327    state_space_llt
        0                knee, cartilage, acl, ligament, cruciate, tibial            0.001318    state_space_llt
        3         shoulder, cuff, rotator, rotator cuff, elbow, fractures            0.000503    state_space_llt

strongest FALLING (by slope_share_per_yr):
 topic_id                                                            label  slope_share_per_yr           model
       10 coronary, heart, myocardial, cardiac, heart failure, ventricular           -0.000804 state_space_llt
       32               sepsis, shock, septic, septic shock, blood, levels           -0.000703 state_space_llt
        9       

### 2a. Small-multiples grid — strongest rising (top row) vs falling (bottom row)

Top row = the 4 topics with the **largest positive** `slope_share_per_yr`; bottom row = the 4 with the **largest negative**. Selection is computed live from the summary.

In [3]:
fig, axes = plt.subplots(2, 4, figsize=(16.5, 8.0), sharex=True)
for ax, tid in zip(axes[0], rising["topic_id"], strict=False):
    fan_chart(ax, int(tid), compact=True)
for ax, tid in zip(axes[1], falling["topic_id"], strict=False):
    fan_chart(ax, int(tid), compact=True)
# Single shared legend (handles from the first panel).
h, lbl = axes[0, 0].get_legend_handles_labels()
fig.legend(
    h, lbl, loc="upper center", ncol=3, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.02)
)
fig.suptitle(
    "(a) Per-topic share trajectories — strongest RISING (top) vs FALLING (bottom)\n"
    "solid = observed share (1995-2025); dashed = projected mean (2026-2030); "
    "shaded = projection band. "
    "Descriptive, exploratory.",
    fontsize=11,
    y=1.07,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_05_fanchart_grid.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print("saved figures/v2_05_fanchart_grid.png")

saved figures/v2_05_fanchart_grid.png


### 2b. Standalone fan charts — headline rising & headline falling topics

Larger single-topic views of the **#1 rising** and **#1 falling** topic by slope (again selected live), with the legend shown.

In [4]:
top_rise = int(rising.iloc[0]["topic_id"])
top_fall = int(falling.iloc[0]["topic_id"])

fig, ax = plt.subplots(figsize=(9.5, 5.6))
fan_chart(ax, top_rise, show_legend=True)
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_05_fan_rising.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"saved figures/v2_05_fan_rising.png  (topic {top_rise})")

fig, ax = plt.subplots(figsize=(9.5, 5.6))
fan_chart(ax, top_fall, show_legend=True)
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_05_fan_falling.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"saved figures/v2_05_fan_falling.png  (topic {top_fall})")

saved figures/v2_05_fan_rising.png  (topic 2)
saved figures/v2_05_fan_falling.png  (topic 10)


## 3. Share vs. volume — the corpus-growth confound

Why share is the headline metric. The two panels below plot the **same illustrative topic** two ways — its **share** of annual assigned output (left) and its **raw volume** of papers (right). Where the two disagree, the gap *is* the corpus-growth confound: raw volume can rise (or hold) simply because the whole 78-journal corpus grew, while share — papers ÷ all-assigned-that-year — controls for it and reveals the topic's true relative trajectory.

> **Partial-2026 note.** 2026 was incomplete at harvest (~**3,872** papers vs the ~**28k/yr** typical of complete years), so **2026 is excluded from fitting** and instead appears as the **first projected year** (2026–2030). That is why the observed series stops at the last complete year, **2025**, and the dashed projection begins at 2026 — the partial year is never treated as an observation.

In [5]:
# Illustrative topic = the headline rising topic (most visible share/volume contrast).
illus = top_rise
row = SUM_IDX.loc[illus]
sub = series[series["topic_id"] == illus].sort_values("year")
obs = sub[sub["kind"] == "observed"]
proj = sub[sub["kind"] == "projected"]
last = obs.iloc[-1]


def _anchor(col):
    return (
        np.concatenate([[last["year"]], proj["year"].to_numpy()]),
        np.concatenate([[last[col]], proj[col].to_numpy()]),
    )


fig, (axL, axR) = plt.subplots(1, 2, figsize=(14.5, 5.4))

# Left: share.
axL.plot(obs["year"], obs["share"], "-o", color=OBS_C, lw=1.7, ms=3.2, label="observed")
px, pm = _anchor("share")
axL.plot(px, pm, "--", color=PROJ_C, lw=1.7, label="projected")
_, plo = _anchor("share_lo")
_, phi = _anchor("share_hi")
axL.fill_between(px, plo, phi, color=PROJ_C, alpha=0.18, lw=0, label="projection band")
axL.axvline(last["year"], color="#bbbbbb", ls=":", lw=0.8)
axL.set_ylim(bottom=0)
axL.set_title("(b1) SHARE of annual assigned output\n(controls for corpus growth)", fontsize=9)
axL.set_xlabel("year")
axL.set_ylabel("share")
axL.legend(loc="best", fontsize=8, frameon=False)

# Right: volume.
axR.plot(obs["year"], obs["volume"], "-o", color=OBS_C, lw=1.7, ms=3.2, label="observed")
vx, vm = _anchor("volume")
axR.plot(vx, vm, "--", color=PROJ_C, lw=1.7, label="projected")
_, vlo = _anchor("volume_lo")
_, vhi = _anchor("volume_hi")
axR.fill_between(vx, vlo, vhi, color=PROJ_C, alpha=0.18, lw=0, label="projection band")
axR.axvline(last["year"], color="#bbbbbb", ls=":", lw=0.8)
axR.set_ylim(bottom=0)
axR.set_title("(b2) RAW VOLUME of papers\n(confounded by corpus growth)", fontsize=9)
axR.set_xlabel("year")
axR.set_ylabel("papers assigned")
axR.legend(loc="best", fontsize=8, frameon=False)

fig.suptitle(
    f"(b) Share vs. volume for topic #{illus}: {row['label']}  ({row['direction']})\n"
    "observed stops at the last complete year (2025); 2026-2030 projected "
    "(partial 2026 excluded from fitting).",
    fontsize=10,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_05_share_vs_volume.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"saved figures/v2_05_share_vs_volume.png  (topic {illus})")

saved figures/v2_05_share_vs_volume.png  (topic 2)


## 4. Rising / falling leaderboards

Top-15 topics by `slope_share_per_yr` in each direction, with the last observed share, the 2030 projected share and its band, the model used, and the assigned `direction` label. Slope is **share per year** (e.g. `+0.0013/yr` ≈ +0.13 share-points of annual output per year). `loglinear_fallback` rows are the 3 topics where the state-space smoother did not converge or had too few observed years and a log-linear fit was substituted.

In [6]:
LB_COLS = [
    "topic_id",
    "label",
    "slope_share_per_yr",
    "last_obs_share",
    "proj_share",
    "proj_share_lo",
    "proj_share_hi",
    "model",
    "direction",
]
ROUND = {
    "slope_share_per_yr": 5,
    "last_obs_share": 4,
    "proj_share": 4,
    "proj_share_lo": 4,
    "proj_share_hi": 4,
}

lb_rising = (
    summary.sort_values("slope_share_per_yr", ascending=False)
    .head(15)[LB_COLS]
    .round(ROUND)
    .reset_index(drop=True)
)
lb_falling = (
    summary.sort_values("slope_share_per_yr", ascending=True)
    .head(15)[LB_COLS]
    .round(ROUND)
    .reset_index(drop=True)
)

print("=== TOP-15 RISING topics (largest positive share slope) ===")
display(lb_rising)
print("=== TOP-15 FALLING topics (largest negative share slope) ===")
display(lb_falling)

=== TOP-15 RISING topics (largest positive share slope) ===


,topic_id,label,slope_share_per_yr,last_obs_share,proj_share,proj_share_lo,proj_share_hi,model,direction
0,2,"health, information, data, care, digital, lear...",0.00331,0.1389,0.1553,0.1101,0.2144,loglinear_fallback,rising
1,13,"covid19, influenza, sarscov2, coronavirus, vac...",0.00133,0.0196,0.0267,0.0058,0.1136,state_space_llt,rising
2,0,"knee, cartilage, acl, ligament, cruciate, tibial",0.00132,0.0575,0.0532,0.0418,0.0675,state_space_llt,rising
3,3,"shoulder, cuff, rotator, rotator cuff, elbow, ...",0.00050,0.0268,0.0297,0.0241,0.0366,state_space_llt,rising
4,19,"pancreatic, pancreatectomy, resection, pancrea...",0.00044,0.0190,0.0225,0.0164,0.0307,state_space_llt,rising
5,1,"colorectal, resection, liver, cancer, rectal, ...",0.00042,0.0355,0.0325,0.0278,0.0380,state_space_llt,rising
6,35,"infection, periprosthetic, ssi, arthroplasty, ...",0.00037,0.0115,0.0136,0.0099,0.0187,state_space_llt,rising
7,72,"opioid, pain, opioid use, opioids, postoperati...",0.00033,0.0073,0.0095,0.0046,0.0193,state_space_llt,rising
8,55,"training, skills, surgical, residents, laparos...",0.00027,0.0070,0.0061,0.0041,0.0090,state_space_llt,rising
9,59,"quantum, topological, spin, magnetic, optical,...",0.00026,0.0095,0.0127,0.0057,0.0282,state_space_llt,rising


=== TOP-15 FALLING topics (largest negative share slope) ===


,topic_id,label,slope_share_per_yr,last_obs_share,proj_share,proj_share_lo,proj_share_hi,model,direction
0,10,"coronary, heart, myocardial, cardiac, heart fa...",-0.00080,0.0103,0.0086,0.0064,0.0114,state_space_llt,falling
1,32,"sepsis, shock, septic, septic shock, blood, le...",-0.00070,0.0019,0.0012,0.0008,0.0016,state_space_llt,falling
2,9,"renal, kidney, glomerular, dialysis, kidney di...",-0.00063,0.0114,0.0095,0.0065,0.0137,state_space_llt,falling
3,7,"dna, rna, transcription, protein, chromatin, r...",-0.00054,0.0134,0.0118,0.0092,0.0150,state_space_llt,falling
4,29,"pylori, colitis, gastric, intestinal, crohns, ...",-0.00049,0.0058,0.0047,0.0039,0.0056,state_space_llt,falling
5,45,"liver, bile, hepatic, hepatocytes, mice, rats",-0.00045,0.0032,0.0020,0.0016,0.0025,state_space_llt,falling
6,20,"hiv, hiv1, immunodeficiency, human immunodefic...",-0.00044,0.0068,0.0060,0.0052,0.0068,state_space_llt,falling
7,6,"valve, aortic, ventricular, mitral, aortic val...",-0.00043,0.0103,0.0064,0.0048,0.0086,state_space_llt,falling
8,11,"ventilation, icu, pressure, respiratory, inten...",-0.00042,0.0110,0.0098,0.0079,0.0121,state_space_llt,falling
9,27,"coronary, artery, cabg, coronary artery, bypas...",-0.00042,0.0050,0.0039,0.0028,0.0055,state_space_llt,falling


## 5. Summary — descriptive, exploratory, and explicitly *not* F3

**What this layer is.** A **descriptive trend projection** for each of the 149 leaf topics: take the topic's annual **share** of assigned output (1995–2025), fit a state-space local-linear-trend model (`state_space_llt`, 146/149; `loglinear_fallback` for the 3 non-converging topics), and **carry the trend forward** to 2026–2030 with uncertainty bands. **All 149 topics fit (`fit_ok` 149/149).** Direction labels split **flat 81 / falling 38 / rising 30**.

**What this layer is NOT.** It is **not a predictive claim** and **not the F3 GNN emergence classifier**, which was the formal forecasting object of V1 and was **signed NULL at Gate G4** (sealed test AUC 0.781 < 0.804 no-graph baseline; H1 FAIL; Brier Wilcoxon favored the baseline). Nothing here re-litigates that. This is a cartographic *where-has-it-been-trending* overlay with stated uncertainty — a reading aid for the map, not a forecast we stand behind.

**Caveats.**
1. **Share, not volume, is the headline** — raw volume is confounded by 78-journal corpus growth (§3). A topic can grow in papers while *losing* share; share is the honest relative signal, volume is shown only to expose the confound.
2. **1995 left-censoring** — the corpus starts in 1995, so early-1990s and prior dynamics are invisible; a topic already mature in 1995 enters mid-trajectory and its slope reflects only the observed window.
3. **Partial-2026 dropped** — 2026 (~3,872 papers vs ~28k/yr) is excluded from fitting and shown as the first *projected* year; the last **observed** year is **2025**.
4. **Band behaviour with horizon** — bands generally **widen** with the projection horizon in the model's transformed (e.g. log/logit) space, but can **compress near the 0/1 share boundary** for saturating topics, so a narrowing band near the floor/ceiling is an artefact of the boundary, not rising confidence. Volume bands can fan out sharply (e.g. the COVID topic's 2030 volume band spans an order of magnitude) — another reason share is the primary read.
5. **Slope is a single global trend** — the LLT level/slope is summarized to one `slope_share_per_yr`; genuinely non-monotone topics (a rise-then-fall like COVID) are flattened by that summary, so always read the **fan chart**, not the slope alone.